In [1]:
from pathlib import Path
import json

from sklearn import metrics
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import skew, skewtest, kurtosis, kurtosistest
from sklearn.metrics.pairwise import cosine_similarity

from lib.pose_utils import get_poem_embedding
from mime_db import MimeDb

# Connect to the database
db = await MimeDb.create()

In [ ]:
# This cell is just to explore the data available for one video -- usually skipped

VIDEO_FILE = "A_Letter_to_My_Nephew.mp4"  # Just the name of the video file, no path

video_path = Path("videos", VIDEO_FILE)

# Get video metadata
video_name = video_path.name
print(video_name)
video_id = await db.get_video_id(video_name)

print("VIDEO ID", video_id)

video_data = await db.get_video_by_id(video_id)
video_fps = video_data["fps"]
video_frame_count = video_data["frame_count"]

video_seconds = video_frame_count / video_fps

video_poses = await db.get_pose_data_from_video(video_id)
video_hands = await db.get_hand_data_from_video(video_id)

In [2]:
def get_distribution_stats(distrib, plot=False):

    if len(distrib) == 0:
        return {
            "count": 0,
            "mean": 0,
            "median": 0,
            "stdev": 0,
            "skewness": 0,
            "kurtosis": 0,
        }

    if skewtest(distrib).pvalue < 0.05:
        skewness = skew(distrib)
    else:
        skewness = 0

    if kurtosistest(distrib).pvalue < 0.05:
        kurtosis_value = kurtosis(distrib)
    else:
        kurtosis_value = 0

    if plot:
        plt.hist(distrib, bins="auto")  # arguments are passed to np.histogram
        plt.show()

    return {
        "count": len(distrib),
        "mean": np.mean(distrib),
        "median": np.median(distrib),
        "stdev": np.std(distrib),
        "skewness": skewness,
        "kurtosis": kurtosis_value,
    }


stylometry_videos = [
    "A_Letter_to_My_Nephew.mp4",
    "Analogy-Dora_Tramontane.mp4",
    "A_Quarelling_Pair-2007_upscaled.mp4",
    "BillTJones_Analogy_Ambros_The_Emigrant.mp4",
    "D-ManintheWaters.mp4",
    "FondlyDoWeHope-Fervently_Do_We_Pray.mp4",
    "Holzer_DuetTruisms_upscaled.mp4",
    "PlayandPlay.mp4",
    "Secret_Pastures.mp4",
    "Story-Time.mp4",
    "WeShallNotBeMoved_upscaled.mp4",
    "A_Balsa_da_Medusa-oratorio_de_Hans_Werner_Henze_Opera_Holandesa.mp4",
    "Democracy_in_America_upscaled.mp4",
    "Don_Giovanni_Mozart.mp4",
    "Go_Down_Moses_2014.mp4",
    "Inferno_Dante.mp4",
    "Parsifal-Wagner-La_Monnaie-De_Munt.mp4",
    "Purgatorio-Dante_upscaled.mp4",
    "Requiem-Mozart-Festival_dAix-en-Provence.mp4",
    "Resurrection-Mahler.mp4",
    "The_Magic_Flute_Mozart_La_Monnaie_De_Munt.mp4",
    "Bluebeard_Voix_Humaine.mp4",
    "Die_Gezeichneten.mp4",
    "Elektra_Strauss.mp4",
    "Ifigenia_emTauris-Gluck.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp1.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp2.mp4",
    "Medeia-Luigi_Cherubini.mp4",
    "Tales_of_Hoffman.mp4",
    "the_french.mp4",
    "Wozzeck_Alban_Berg.mp4",
]

director_videos = {}
video_directors = {}
director_videos["BillTJones"] = [
    "A_Letter_to_My_Nephew.mp4",
    "Analogy-Dora_Tramontane.mp4",
    "A_Quarelling_Pair-2007_upscaled.mp4",
    "BillTJones_Analogy_Ambros_The_Emigrant.mp4",
    "D-ManintheWaters.mp4",
    "FondlyDoWeHope-Fervently_Do_We_Pray.mp4",
    "Holzer_DuetTruisms_upscaled.mp4",
    "PlayandPlay.mp4",
    "Secret_Pastures.mp4",
    "Story-Time.mp4",
    "WeShallNotBeMoved_upscaled.mp4",
]
director_videos["Castellucci"] = [
    "A_Balsa_da_Medusa-oratorio_de_Hans_Werner_Henze_Opera_Holandesa.mp4",
    "Democracy_in_America_upscaled.mp4",
    "Don_Giovanni_Mozart.mp4",
    "Go_Down_Moses_2014.mp4",
    "Inferno_Dante.mp4",
    "Parsifal-Wagner-La_Monnaie-De_Munt.mp4",
    "Purgatorio-Dante_upscaled.mp4",
    "Requiem-Mozart-Festival_dAix-en-Provence.mp4",
    "Resurrection-Mahler.mp4",
    "The_Magic_Flute_Mozart_La_Monnaie_De_Munt.mp4",
]
director_videos["Warlikowski"] = [
    "Bluebeard_Voix_Humaine.mp4",
    "Die_Gezeichneten.mp4",
    "Elektra_Strauss.mp4",
    "Ifigenia_emTauris-Gluck.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp1.mp4",
    "Lady_Macbeth_of_Mtsensk_Districtp2.mp4",
    "Medeia-Luigi_Cherubini.mp4",
    "Tales_of_Hoffman.mp4",
    "the_french.mp4",
    "Wozzeck_Alban_Berg.mp4",
]
director_videos["Other"] = []
director_ids = {"BillTJones": 0, "Castellucci": 1, "Warlikowski": 2}
stylometry_directors = ["BillTJones", "Castellucci", "Warlikowski"]

stylometry_video_ids = {}
for video_name in stylometry_videos:
        video_record = await db.get_video_by_name(video_name)
        video_uuid = video_record.get("id", None)
        stylometry_video_ids[video_name] = str(video_uuid)

for director in director_videos:
    for video_name in director_videos[director]:
        video_directors[video_name] = director_ids[director]

video_labels = [video_directors[video_name] for video_name in stylometry_videos]

with open("archetypes/pose_archetypes.json", "r") as poses_file:
    pose_archetypes = json.load(poses_file)
with open("archetypes/hand_archetypes.json", "r") as hands_file:
    hand_archetypes = json.load(hands_file)

pose_descriptions = [arch["description"] for arch in pose_archetypes] # [:10] # Only use 3x3 + 1 poses

hand_descriptions = [arch["description"] for arch in hand_archetypes]


In [3]:
# Used to add the pose embeddings to the pose archetypes (they're originally just
# available as global 3D coords) and export them to a JSON file to be used at
# various places in the site.

import copy

full_pose_archetypes = copy.deepcopy(pose_archetypes)

for a, archetype in enumerate(pose_archetypes):
    flattened_global3d = [c for i, c in enumerate(archetype["global3d_coco13"]) if (i + 1) % 3 != 0]
    poem_embedding = get_poem_embedding(flattened_global3d)
    full_pose_archetypes[a]["poem_embedding"] = poem_embedding

#with open("full_pose_archetypes.json", "w", encoding="utf-8") as outfile:
#    json.dump(full_pose_archetypes, outfile, indent=4)

pose_archetypes = full_pose_archetypes

In [4]:
# Get the prevalence of a pose archetype in a video
# Given a pose archetype
# For every pose in the video
#   Calculate its cosine similarity to the archetype, put it into an array
# Average the similarities (maybe derive some other statistics too)
def get_fingerprints(archetypes, video_poses_or_hands, comparison_metric):
    mean_sims = [] # Could do medians as well/instead, but it doesn't make much difference
    for archetype in archetypes: # [:10]
        sims = []
        if comparison_metric == "poem_embedding":
            flattened_global3d = [c for i, c in enumerate(archetype["global3d_coco13"]) if (i + 1) % 3 != 0]
            archetype_vector = get_poem_embedding(flattened_global3d)
        else:
            archetype_vector = archetype[comparison_metric]
        for pose_or_hand in video_poses_or_hands:
            sims.append(1 - cosine(archetype_vector, pose_or_hand[comparison_metric]))

        mean_sims.append(np.mean(sims))

    return mean_sims

if not os.path.isfile("archetypes/archetype_fingerprints.json"):
    delsarte_pose_fingerprints = {}
    for video_name in stylometry_videos:
        print("Computing pose fingerprints for", video_name)
        video_id = await db.get_video_id(video_name)
        video_poses = await db.get_pose_data_from_video(video_id)
        delsarte_pose_fingerprints[str(video_id)] = {"video_name": video_name, "poem_embedding": get_fingerprints(pose_archetypes, video_poses, "poem_embedding"), "global3d_coco13": get_fingerprints(pose_archetypes, video_poses, "global3d_coco13")}

    delsarte_hand_fingerprints = {}
    for video_name in stylometry_videos:
        print("Computing hand fingerprints for", video_name)
        video_id = await db.get_video_id(video_name)
        video_hands = await db.get_hand_data_from_video(video_id)
        delsarte_hand_fingerprints[str(video_id)] = {"video_name": video_name, "joint_angles3d": get_fingerprints(hand_archetypes, video_hands, "joint_angles3d"), "class_weights": get_fingerprints(hand_archetypes, video_hands, "class_weights")}
else:
    with open("archetypes/archetype_fingerprints.json", "r", encoding="utf-8") as archfile:
        archetype_data = json.load(archfile)
        delsarte_pose_fingerprints = archetype_data["poses"]
        delsarte_hand_fingerprints = archetype_data["hands"]

In [5]:
all_fingerprints = {"poses": delsarte_pose_fingerprints, "hands": delsarte_hand_fingerprints}

#with open("archetypes/archetype_fingerprints.json", "w", encoding="utf-8") as outfile:
#    json.dump(all_fingerprints, outfile, indent=4)

In [6]:
# This determines whether the cells below consider poses or hands
delsarte_fingerprints = delsarte_pose_fingerprints
# delsarte_fingerprints = delsarte_hand_fingerprints

In [ ]:
# "Dumbest" possible classifier, using "leave one out" approach and cosine similarity

hits = 0
misses = 0

predictions = []

metric = "class_weights" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

for i, video_name in enumerate(stylometry_videos):

    fingerprints_by_director = {
        "BillTJones": [],
        "Castellucci": [],
        "Warlikowski": [],
    }

    video_id = stylometry_video_ids[video_name]

    for e, comparison_video in enumerate(stylometry_videos):
        if e == i:
            continue

        video_director = stylometry_directors[video_directors[comparison_video]]
        comparison_video_id = stylometry_video_ids[comparison_video]
        
        fingerprints_by_director[video_director].append(delsarte_fingerprints[comparison_video_id][metric])

    billtjones_avg_fingerprint = np.mean(
        fingerprints_by_director["BillTJones"], axis=0
    )
    castellucci_avg_fingerprint = np.mean(
        fingerprints_by_director["Castellucci"], axis=0
    )
    warlikowski_avg_fingerprint = np.mean(
        fingerprints_by_director["Warlikowski"], axis=0
    )

    directors_matrix = np.stack(
        (
            billtjones_avg_fingerprint,
            castellucci_avg_fingerprint,
            warlikowski_avg_fingerprint,
        ),
        axis=0,
    )

    video_fingerprint = delsarte_fingerprints[video_id][metric]
    true_label = video_directors[video_name]

    sims = cosine_similarity(directors_matrix, np.array([video_fingerprint]))

    pred_label = np.argmax(sims)
    print(video_name, "Prediction:", pred_label, "True:", true_label)

    predictions.append(pred_label)

    if pred_label == true_label:
        hits += 1
    else:
        misses += 1

print("Hits:", hits, "Misses:", misses, f"{hits*100/(hits+misses):.2f}%")

confusion_matrix = metrics.confusion_matrix(video_labels, predictions)
cm_display = metrics.ConfusionMatrixDisplay.from_predictions(
    video_labels,
    predictions,
    display_labels=["BillTJones", "Castellucci", "Warlikowski"],
    colorbar=False,
)
plt.title("LOO - pose embedding sims to Delsarte archetypes")
cm_display.plot()
#plt.show()

In [ ]:
# Train a classifier on all but one; see where that one is classified,
# repeat for all videos (leave-one-out).
# Potentially repeat the process for different random seeds, if the
# classification algorithm is nondeterministic

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# import xgboost as xgb
# from sklearn.svm import SVC, LinearSVC, NuSVC
# from sklearn.neural_network import MLPClassifier

N_SEEDS = 10

hits = 0
misses = 0
predicted = []
actual = []

metric = "joint_angles3d" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

X = []
for video_name in stylometry_videos:
    video_id = stylometry_video_ids[video_name]
    X.append(delsarte_fingerprints[video_id][metric])

for r in range(N_SEEDS):
    for h, held_out_vector in enumerate(X):

        X_train = np.delete(X, h, 0)
        y_train = np.delete(video_labels, h, 0)
        # X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.33, random_state=r)

        # clf = MLPClassifier()
        # lf = NuSVC(random_state=42)
        # clf = GaussianNB()
        # clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)
        clf = RandomForestClassifier(random_state=r)
        clf.fit(X_train, y_train)
        # clf.fit(X_train,  y_train, eval_set=[(X_test, y_test)])

        # pred = clf.predict(held_out_vector.reshape(1, -1))[0]
        pred = clf.predict([held_out_vector])[0]

        if pred == video_labels[h]:
            hits += 1
        else:
            misses += 1

        actual.append(video_labels[h])
        predicted.append(pred)

        # print("For ", video_names[h], "pred is", pred, "true is", video_labels[h])

print(f"Hits: {hits}, Misses: {misses}, {hits*100/(hits+misses):.2f}%")

confusion_matrix = metrics.confusion_matrix(actual, predicted)
cm_display = metrics.ConfusionMatrixDisplay.from_predictions(
    actual,
    predicted,
    display_labels=["BillTJones", "Castellucci", "Warlikowski"],
    colorbar=False,
)
plt.title("RF - Global 3D COCO-13 sims to Delsarte archetypes")
cm_display.plot()

In [7]:
from sklearn.model_selection import cross_validate, cross_val_score, KFold
import random
from datetime import datetime

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Another way to do it
# scoring = ['precision_macro', 'recall_macro']
# clf = RandomForestClassifier(random_state=0)
# scores = cross_validate(clf, X, video_labels, scoring=scoring)
# print("precision scores:", scores["test_precision_macro"], "recall scores:", scores["test_recall_macro"])

metric = "poem_embedding" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

X = []
for video_name in stylometry_videos:
    video_id = stylometry_video_ids[video_name]
    X.append(delsarte_fingerprints[video_id][metric])

N_SEEDS = 10

print("10-fold cross validation with 10 random seeds for video feature vectors")
all_scores = []
for i in range(N_SEEDS):
    r = random.seed(datetime.now().timestamp())
    # clf = RandomForestClassifier(random_state=r)
    clf = GaussianNB()

    # clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)

    cv = KFold(n_splits=10, random_state=r)
    scores = cross_val_score(
        clf, X, video_labels, scoring="accuracy", cv=cv, n_jobs=-1
    )
    all_scores.extend(scores)
print(
    "Accuracy: %.3f ,\nStandard Deviations :%.3f"
    % (np.mean(all_scores), np.std(all_scores))
)

In [ ]:
from sklearn.inspection import permutation_importance
import time

feature_names = pose_descriptions # or pose_descriptions

metric = "poem_embedding" # either poem_embedding or global3d_coco13 for poses, or joint_angles3d or class_weights for hands

X = []
for video_name in stylometry_videos:
    video_id = stylometry_video_ids[video_name]
    X.append(delsarte_fingerprints[video_id][metric])

import xgboost as xgb

X_train, X_test, y_train, y_test = train_test_split(
    X, video_labels, test_size=0.33, random_state=42
)

clf = RandomForestClassifier(random_state=0)
# clf = GaussianNB()
# clf = xgb.XGBClassifier(tree_method="hist", early_stopping_rounds=2)

clf.fit(X_train, y_train)
# clf.fit(X_train, y_train, eval_set=[(X_test, y_test)])

start_time = time.time()
perm_importance = permutation_importance(
    clf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2
)
elapsed_time = time.time() - start_time
print(f"Elapsed time to compute the importances: {elapsed_time:.3f} seconds")

clf_importances = pd.Series(perm_importance.importances_mean, index=feature_names)

# importances = clf.feature_importances_
# importances_std = np.std([tree.feature_importances_ for tree in clf.estimators_], axis=0)
# clf_importances = pd.Series(importances, index=vector_names)

fig, ax = plt.subplots()
clf_importances.plot.bar(yerr=perm_importance.importances_std, ax=ax)
ax.set_title("Archetype importances (permutation) - pose embedding")
ax.set_ylabel("Mean accuracy decrease")
ax.tick_params(axis="x", labelrotation=40)
plt.setp(ax.xaxis.get_majorticklabels(), ha="right")
fig.tight_layout()
plt.show()

sorted_idx = perm_importance.importances_mean.argsort()

# plt.barh(
#     np.array(feature_names)[sorted_idx],
#     perm_importance.importances_mean[sorted_idx],
#     xerr=perm_importance.importances_std[sorted_idx],
# )
# plt.barh(np.array(vector_names)[sorted_idx], perm_importance.importances_mean[sorted_idx], xerr=perm_importance.importances_std[sorted_idx])
plt.barh(
    np.array(feature_names)[sorted_idx],
    perm_importance.importances_mean[sorted_idx],
    xerr=perm_importance.importances_std[sorted_idx],
)
plt.xlabel("Permutation Importance")
plt.show()

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fp_type = "poses"
delsarte_fingerprints = delsarte_pose_fingerprints
descriptions = pose_descriptions
metric = "poem_embedding" # pose: global3d_coco13 or poem_embedding; hand: joint_angles3d or class_weights
archetypes = pose_archetypes
performance = "Don_Giovanni_Mozart.mp4"

def offset_image(x, y, arch, bar_is_too_short, ax):
    img = plt.imread(f"archetypes/{fp_type}/{arch['image_filename']}")
    im = OffsetImage(img, zoom=.09, cmap=mpl.colormaps['gray'])
    im.image.axes = ax
    x_offset = .2
    if bar_is_too_short:
        x = 0
    ab = AnnotationBbox(im, (x, y), xybox=(x_offset, 0), frameon=False,
                        xycoords='data', boxcoords="offset points", pad=0)
    ax.set_facecolor('#FFFFFF')
    ax.add_artist(ab)

fig = plt.figure(figsize=(8,12))

height = .5

fingerprint_data = list(reversed(delsarte_fingerprints[performance][metric]))
plt.barh(list(reversed(descriptions)), fingerprint_data, height=height, align='center', alpha=0.8, color="#663399")
ax = plt.gca()
ax.tick_params(axis="y", labelrotation=40)

max_value = max(fingerprint_data)

for a, arch in enumerate(list(reversed(archetypes))):
    img = plt.imread(f"archetypes/{fp_type}/{arch['image_filename']}")
    value = fingerprint_data[a]
    offset_image(value, a, arch, bar_is_too_short=value < max_value / 10, ax=plt.gca())
    
plt.subplots_adjust(left=0.15)

plt.xlim(0, max(fingerprint_data) * 1.10)
plt.ylim(-0.5, len(descriptions) - 0.5)
plt.tight_layout()

plt.show()


In [9]:
import pickle

# If derived data has already been calculated, load it from the pkl files
movelet_poem_embeds_by_video = pickle.load(
    open("stylometry/poem_embeds_by_video.pkl", "rb")
)
global_3d_coco_13_by_video = pickle.load(
    open("stylometry/global_3d_coco_13_by_video.pkl", "rb")
)
action_embeds_by_video = pickle.load(open("stylometry/action_embeds_by_video.pkl", "rb"))

# Load cached stats
all_video_stats = json.load(open("stylometry/video_stats.json", "r", encoding="utf-8"))

# Augment video data with directors and save stats to files
for video_name in all_video_stats:
    if video_name.startswith("Don_Giovanni-"):
        continue

    all_video_stats[video_name]["director"] = video_directors[video_name]

In [11]:
# Group poem and action embeddings and 3D global poses by director for embedding analysis

poem_embeddings_by_director = {}
action_embeddings_by_director = {}
global_3d_coco_13_by_director = {}

stylometry_directors = ["BillTJones", "Castellucci", "Warlikowski"]

for director in stylometry_directors:
    if director == "Other":
        continue

    poem_embeddings_by_director[director] = []
    action_embeddings_by_director[director] = []
    global_3d_coco_13_by_director[director] = []

    for video_name in director_videos[director]:
        poem_embeddings_by_director[director].extend(
            movelet_poem_embeds_by_video[video_name]
        )
        action_embeddings_by_director[director].extend(
            action_embeds_by_video[video_name]
        )
        global_3d_coco_13_by_director[director].extend(
            global_3d_coco_13_by_video[video_name]
        )

billtjones_mean_poem_embedding = np.mean(
    poem_embeddings_by_director["BillTJones"], axis=0
)
castellucci_mean_poem_embedding = np.mean(
    poem_embeddings_by_director["Castellucci"], axis=0
)
warlikowski_mean_poem_embedding = np.mean(
    poem_embeddings_by_director["Warlikowski"], axis=0
)

label_colormap = {0: "red", 1: "green", 2: "blue", 3: "orange"}
label_colors = [label_colormap[l] for l in video_labels]

In [81]:
import umap
import matplotlib.patches as mpatches

from matplotlib.offsetbox import OffsetImage, AnnotationBbox

IMAGE_SIDE = 30

def getImage(path, dims=(IMAGE_SIDE, IMAGE_SIDE)):
    im = PIL.Image.open(path)
    im = im.resize(dims, resample=PIL.Image.Resampling.LANCZOS)
    return OffsetImage(im, cmap=mpl.colormaps['gray'], dpi_cor=True, resample=True)


# Project large numbers of embeddings into 2D, attempt to colorize by director
# and highlight where directors' (and specific videos'?) avg embeddings are.
# NOTE: Running on the full embeddings set would exceed the RAM of most (maybe all)
# machines, so we sample by SAMPLE_RATE.
# Note also that there are almost no identical embeddings, and their dimensionality
# is such that rounding the values and removing duplicates only decreases their size
# by a tiny fraction.

SAMPLE_RATE = 100  # 1/SAMPLE_RATE poses will be used

# For use with the Tensorflow Embedding Projector or similar visualizations
# poem_metadata = open("stylometry/sampled_pose_metadata.tsv", "w")
# poem_metadata.write("Director\tWork\n")

all_poem_embeddings = []
embedding_label_colors = []

# e = 0
for dir_name in ["BillTJones", "Castellucci", "Warlikowski"]:
    for video_name in director_videos[dir_name]:
        work_embeds = movelet_poem_embeds_by_video[video_name]
        # for embed in work_embeds:
        #     if e % SAMPLE_RATE == 0:
        #         poem_metadata.write(f"{dir_name}\t{video_name}\n")
        #     e += 1

i = 0
for emb in poem_embeddings_by_director["BillTJones"]:
    if i % SAMPLE_RATE == 0:
        all_poem_embeddings.append(emb)
        embedding_label_colors.append(label_colormap[0])
    i += 1
for emb in poem_embeddings_by_director["Castellucci"]:
    if i % SAMPLE_RATE == 0:
        all_poem_embeddings.append(emb)
        embedding_label_colors.append(label_colormap[1])
    i += 1
for emb in poem_embeddings_by_director["Warlikowski"]:
    if i % SAMPLE_RATE == 0:
        all_poem_embeddings.append(emb)
        embedding_label_colors.append(label_colormap[2])
    i += 1

all_poem_embeddings.extend(
    [
        billtjones_mean_poem_embedding,
        castellucci_mean_poem_embedding,
        warlikowski_mean_poem_embedding,
    ]
)

for archetype in pose_archetypes:
    all_poem_embeddings.append(archetype["poem_embedding"])

print("CALCULATING UMAP PROJECTION ON", len(all_poem_embeddings), "POEM embeddings")

clusterable_embedding = umap.UMAP(
    metric="cosine",
    n_neighbors=25,
    min_dist=0,
    n_components=2,
    n_jobs=8,
).fit_transform(all_poem_embeddings)

plt.figure(figsize=(10, 10))

# XXX This gets messy...
# all but the last 19 embeddings are the individual sampled pose embeddings for each video,
# followed by the average embedding for each director (n=3) and for each archetype (n=16)

# Plots markers for all of the extracted "key" poses from all the videos
plt.scatter(
    clusterable_embedding[:-19, 0],
    clusterable_embedding[:-19, 1],
    s=2,
    alpha=0.3,
    c=embedding_label_colors,
)
# Plots color-coded hexagonal markers for the 3 directors
plt.scatter(
    clusterable_embedding[-19:-16, 0],
    clusterable_embedding[-19:-16, 1],
    s=200,
    marker="H",
    c=["red", "green", "blue"],
    edgecolors=["black", "black", "black"],
)
# Can be used to plot orange hexagonal markers for all of the archetypes
# plt.scatter(
#     clusterable_embedding[-16:, 0],
#     clusterable_embedding[-16:, 1],
#     s=200,
#     marker="H",
#     c=["orange"] * 16,
#     edgecolors=["black"] * 16,
# )

# Draws the archetypes where their embeddings are placed (beware overlap)
# along with their label texts (could be truncated/shortened in some cases)
archetype_clusters = []

for a in range(len((pose_archetypes))):
    a_x = clusterable_embedding[-16 + a, 0] * 100
    a_y = clusterable_embedding[-16 + a, 1] * 100

    for b in range(len((pose_archetypes))):
        if a >= b:
            continue
        
        already_in_cluster = False

        for cluster in archetype_clusters:
            if b in cluster:
                already_in_cluster = True
                break

        if already_in_cluster:
            continue

        b_x = clusterable_embedding[-16 + b, 0] * 100
        b_y = clusterable_embedding[-16 + b, 1] * 100

        if abs(a_x - b_x) <= IMAGE_SIDE and abs(a_y - b_y) <= IMAGE_SIDE:
            found_match = False
            for i in range(len(archetype_clusters)):
                if a in archetype_clusters[i]:
                    archetype_clusters[i].add(b)
                    found_match = True
                    break
            if not found_match:
                archetype_clusters.append(set([a, b]))
    
for a in range(len((pose_archetypes))):
    in_cluster = False
    for cluster in archetype_clusters:
        if a in cluster:
            in_cluster = True

    if not in_cluster:
        archetype_clusters.append(set([a]))

for cluster in archetype_clusters:

    cluster_centroid = [clusterable_embedding[-16 + list(cluster)[0], 0], clusterable_embedding[-16 + list(cluster)[0], 1]]

    for i, a in enumerate(list(cluster)):

        midpoint = (len(cluster) - 1) / 2
        x_offset = (i - midpoint) / 2
        y_offset = (midpoint - i) / 2

        a_x = clusterable_embedding[-16 + a, 0]
        a_y = clusterable_embedding[-16 + a, 1] + y_offset

        archetype_label = pose_descriptions[a]
        archetype_image_path = "archetypes/poses/" + pose_archetypes[a]["image_filename"]
        ax = plt.gca()
        ab = AnnotationBbox(getImage(archetype_image_path), (a_x, a_y), frameon=True)
        ax.add_artist(ab)
        plt.text(
            a_x,
            a_y,
            archetype_label,
            fontsize="x-small",
        )


# Could be used to plot markers for the individual videos
# for v, video_name in enumerate(stylometry_videos):
#     plt.text(
#         clusterable_embedding[-34 + v, 0],
#         clusterable_embedding[-34 + v, 1],
#         video_name,
#         fontsize="x-small",
#     )

red_patch = mpatches.Patch(color="red", label="BillTJones")
green_patch = mpatches.Patch(color="green", label="Castellucci")
blue_patch = mpatches.Patch(color="blue", label="Warlikowski")
plt.legend(handles=[red_patch, green_patch, blue_patch])

plt.savefig("archetypes_embedding_umap.png", dpi=256)

print(archetype_clusters)